# 🧪 External-data benchmark (competition-compliant)

**Rules this notebook follows** (organisers' clarification): if you train, use only external data; every tuned parameter
is tuned on external data, never on the ground-truth possession data or the provided video. So:

* the **public ground truth is not read anywhere in this notebook**;
* models train on the seven **IDSSE** matches (Sportec/DFL, CC-BY 4.0; the provider's own `BallStatus` / `BallPossession`
  labels) and are validated **leave-one-match-out**: every match is predicted by a model that never saw it;
* decoder settings (smoothing width, minimum possession length, contact filter) are chosen on those external predictions;
  `tuned_on_others` below scores each match with settings picked from the *other* six, an honest transfer estimate;
* the provided match contributes only its tracking: as model input, and to copy where its ball goes unseen (so IDSSE's
  always-present ball has the same gaps).

Metrica's sample games are not used: their README states no licence (confirm with the organisers before adding them).
Caveat that matters: IDSSE's possession definition is more fragmented than the competition's, so absolute scores here are
not predictions of the leaderboard. Use them to compare model families and decoder rules.

Data: `python starter/external.py --fetch-idsse` (2.6 GB, git-ignored). Cite the IDSSE dataset (Bassek et al., figshare
doi 10.6084/m9.figshare.28196177).

In [1]:
import os, sys, time, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "starter" else os.getcwd()
sys.path.insert(0, os.path.join(ROOT, "starter")); sys.path.insert(0, os.path.join(ROOT, "evaluation"))
import possession_metrics as pm
import external as ext
import external_pipeline as ep
pd.set_option("display.width", 200)
if not all(os.path.isfile(p) for m in ext.IDSSE_MATCHES for p in ext.idsse_files(m).values()):
    ext.fetch_idsse()

## 1. Load the seven matches

In [2]:
sets = ep.load_idsse_sets(ep.ball_unseen_pattern(), advanced=True, control_s=3, boundary=True,
                         motion=True, temporal=True, augment=True)  # 109 features, controlled-possession labels, mirrored copies for training
def stats(m, d):
    y = d["y"]; sp = pm.possession_spells(y, pm.valid_runs(y)); L = np.array([(b - a) / 25 for a, b, _ in sp])
    return dict(match=m, frames=len(y), H=(y == 0).mean(), A=(y == 1).mean(), D=(y == 2).mean(),
                per_5min=len(sp) / (len(y) / 25 / 300), median_s=np.median(L), short_lt_4s=(L < 4).mean())
pd.DataFrame([stats(m, d) for m, d in sets.items()]).round(3)

,match,frames,H,A,D,per_5min,median_s,short_lt_4s
0,J03WMX,145967,0.261,0.324,0.415,12.177,9.52,0.101
1,J03WN1,141561,0.257,0.226,0.517,13.722,7.36,0.170
2,J03WOH,137214,0.282,0.243,0.476,12.244,9.28,0.147
3,J03WOY,142536,0.266,0.269,0.465,12.944,8.08,0.134
4,J03WPY,146211,0.349,0.209,0.443,11.798,10.00,0.117
5,J03WQQ,142345,0.256,0.336,0.407,11.908,10.08,0.128
6,J03WR9,146810,0.363,0.218,0.419,10.830,9.36,0.132


## 2. Leave-one-match-out posteriors for each model family

In [3]:
POSTS, t = {}, time.time()
for name in ep.MODELS:
    POSTS[name] = ep.lomo_posteriors(name, sets); print(name, round(time.time() - t), "s")

logreg 151 s


hierarchical 229 s


lightgbm 278 s


## 3. Tune the decoder on external predictions; compare models
Per held-out match: `plain_w25` = smoothing only; `tuned_on_others` = decoder settings picked on the other six matches'
predictions. Means and standard deviations are over the seven matches (a match is the unit of evidence here: roughly
300-500 provider-labelled possessions each).

In [4]:
TUNED, rows, details = {}, [], {}
for name in ep.MODELS:
    joint, table = ep.tune_decoder(POSTS[name], sets); TUNED[name] = joint; details[name] = pd.DataFrame(table)
    d = details[name]
    rows.append(dict(model=name, plain_w25=d.plain_w25.mean(), tuned_on_others=d.tuned_on_others.mean(),
                     sd_over_matches=d.tuned_on_others.std(), joint_decoder=str({k: v for k, v in joint.items() if k != "inflight_thr"})))
board = pd.DataFrame(rows).sort_values("tuned_on_others", ascending=False).reset_index(drop=True)
board.round(3)

,model,plain_w25,tuned_on_others,sd_over_matches,joint_decoder
0,lightgbm,0.594,0.703,0.038,"{'w': 25, 'min_s': 3, 'contact_m': None}"
1,logreg,0.562,0.659,0.039,"{'w': 25, 'min_s': 2, 'contact_m': None}"
2,hierarchical,0.553,0.651,0.038,"{'w': 25, 'min_s': 2, 'contact_m': None}"


In [5]:
best = board.model.iloc[0]
print("selected on external data only:", best)
details[best].round(3)

selected on external data only: lightgbm


,match,tuned_on_others,plain_w25,params
0,J03WMX,0.739,0.618,"{'w': 25, 'min_s': 3, 'contact_m': None}"
1,J03WN1,0.635,0.551,"{'w': 50, 'min_s': 3, 'contact_m': None}"
2,J03WOH,0.696,0.611,"{'w': 25, 'min_s': 3, 'contact_m': None}"
3,J03WOY,0.675,0.574,"{'w': 25, 'min_s': 3, 'contact_m': None}"
4,J03WPY,0.732,0.628,"{'w': 25, 'min_s': 3, 'contact_m': None}"
5,J03WQQ,0.738,0.617,"{'w': 25, 'min_s': 3, 'contact_m': None}"
6,J03WR9,0.702,0.556,"{'w': 25, 'min_s': 3, 'contact_m': None}"


## 3b. Domain weighting (inputs only)
This match's tracking differs from the Bundesliga matches (the ball is unseen far more often, different tempo). A small
classifier learns to tell external frames from this match's frames **using the tracking features alone, no labels**, and
the external frames are weighted by how much they resemble this match. Weights are cross-fitted by match. Applies to the
LightGBM family; kept because it shrinks the input shift (7 strongly shifted features become 1) and costs at most 0.01 of
leave-one-match-out score (with the temporal features it is now 0.004 lower, so its benefit is mainly the smaller shift).

In [6]:
Xt = ep.target_features()
W = ep.domain_weights(sets, Xt)
Xs = np.vstack([d["X"][::6] for d in sets.values()]); ws = np.concatenate([W[m][::6] for m in sets])
smd = lambda mu: np.abs(Xt.mean(0) - mu) / (Xs.std(0) + 1e-9)
before, after = smd(Xs.mean(0)), smd((Xs * ws[:, None]).sum(0) / ws.sum())
print("features shifted by > 0.3 sd:  before %d  after %d   (effective sample size %.0f%%)" %
      ((before > .3).sum(), (after > .3).sum(), 100 * ws.sum() ** 2 / (ws ** 2).sum() / len(ws)))
POSTS_W = ep.lomo_posteriors("lightgbm", sets, W)
joint_w, table_w = ep.tune_decoder(POSTS_W, sets)
pd.DataFrame([dict(model="lightgbm", weighting="none", tuned_on_others=details["lightgbm"].tuned_on_others.mean()),
              dict(model="lightgbm", weighting="domain", tuned_on_others=pd.DataFrame(table_w).tuned_on_others.mean())]).round(3)

features shifted by > 0.3 sd:  before 7  after 1   (effective sample size 39%)


,model,weighting,tuned_on_others
0,lightgbm,none,0.703
1,lightgbm,domain,0.699


## 4. Export the full-match submission
Train the selected family on all seven matches, apply the externally-tuned decoder to this match's two halves, write the
CSV, and validate its format. Then run the official scorer **once** from a terminal, to report the number
(`python evaluation/run_evaluation.py`); nothing above this line depends on it, and it must not be used to pick between
models, weights or settings.

In [7]:
import subprocess
from generate_full_submission import predict_and_write
use_w = best == "lightgbm"      # domain weighting applies to the LightGBM family
params = joint_w if use_w else TUNED[best]
path = predict_and_write(ep.fit(best, sets, weights=W if use_w else None), params, "ext_%s.csv" % best)
print(subprocess.run([sys.executable, os.path.join(ROOT, "evaluation", "validate_submission.py"), path],
                     capture_output=True, text=True).stdout[:900])

wrote /Users/mushroomyyy/more-than-a-hack-2027/evaluation/results/ext_lightgbm.csv


OK    ext_lightgbm.csv
      199319 rows, frame 0..199318

      label counts
        H    61281   30.7%
        A    43415   21.8%
        D    54161   27.2%
        X    40462   20.3%

      reports 235 possessions of mean 17.8 s
      (counted with each X as a gap; the scorer bridges abstentions inside a real
       possession, so its count can be lower than this one)

Valid. Note this checks the format only -- it says nothing about whether the labels are right.
Score it against the public ground truth with run_evaluation.py.

